# Project Example (Reference Solution) — Breast Cancer Classification with PCA

<a target="_blank" href="https://colab.research.google.com/github/LuWidme/uk259/blob/rework/solutions/10_Project_Example_BreastCancer_solutions.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> ⏱️ Reference for **Day 5** · Instructor aid — a complete worked example of the project.

This notebook is a **model solution** for the Day-5 project (`demos/10_Project_Template.ipynb`).
It walks through the **same six milestones** the students follow, using the **Breast Cancer Wisconsin**
dataset and adding **PCA (Principal Component Analysis)** as a preprocessing step.

**For instructors / students who are already done:** this is *one* good way to do the project, not the
only one. The goal is to show the full workflow end-to-end, with short explanations of *why* each step matters.

**The task:** predict whether a tumor is **malignant** (cancerous) or **benign** (harmless) from 30
measurements taken from cell-nucleus images. This is a **binary classification** problem.

In [ ]:
# === COURSE SETUP — run this cell first! ===
%pip install -q numpy pandas matplotlib seaborn scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme()
print("Setup complete — you are ready to go!")

## Milestone 1 — Explore the data

First we **load** the data and get a feel for it: How many samples? How many features?
Are the two classes balanced? What do the numbers look like?

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")   # 0 = malignant, 1 = benign

print("Samples:", X.shape[0], "| Features:", X.shape[1])
print("Target names:", list(data.target_names))   # ['malignant' 'benign']
X.head()

In [ ]:
# Class balance — how many of each?
counts = y.value_counts().rename({0: "malignant", 1: "benign"})
print(counts)

plt.figure(figsize=(5, 4))
sns.countplot(x=y.map({0: "malignant", 1: "benign"}))
plt.title("Class balance")
plt.xlabel("")
plt.show()

The classes are reasonably balanced (a bit more benign than malignant), so plain **accuracy** is an
acceptable metric here — but we will also look at the confusion matrix, because in medicine a *missed
cancer* (false negative) is far worse than a false alarm.

In [ ]:
# Quick look at the value ranges — note how different the scales are
X.describe().T[["mean", "min", "max"]].head(8)

**Important observation:** the features live on very different scales (some are < 1, some are in the
hundreds). PCA and most distance-based models are sensitive to scale, so **we must standardize** before
applying them. We do that in the next milestone.

In [ ]:
# How strongly are some features related? (a small correlation heatmap)
plt.figure(figsize=(8, 6))
sns.heatmap(X.iloc[:, :10].corr(), cmap="coolwarm", center=0)
plt.title("Correlation between the first 10 features")
plt.show()

Many features are **highly correlated** (e.g. *mean radius*, *mean perimeter*, *mean area* all measure
size). That redundancy is exactly what PCA can compress.

## Milestone 2 — Clean & prepare the data

This dataset has **no missing values** (it is a clean, curated dataset), so cleaning is light. The key
preparation steps are:

1. **Train/test split** — hold back 20% to test on data the model never saw.
2. **Standardization** — rescale every feature to mean 0, std 1. We *fit* the scaler on the training data
   only, then apply it to both — otherwise information from the test set leaks into training.

In [ ]:
print("Missing values:", X.isnull().sum().sum())   # 0

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit on TRAIN only
X_test_scaled = scaler.transform(X_test)         # apply to test

print("Train:", X_train_scaled.shape, "| Test:", X_test_scaled.shape)

## Milestone 3 — PCA as a preprocessing step

We have **30 features**, many of them redundant. **PCA** finds new "combined" features (called
*principal components*) that capture as much of the variation as possible, using fewer dimensions.

Intuition: instead of 30 overlapping rulers, PCA builds a handful of new rulers pointing in the directions
where the data actually spreads out. We use it for two things here:

1. **Visualization** — squeeze 30D down to 2D so we can *see* the classes.
2. **Compression** — keep enough components to retain ~95% of the information, then train on those.

In [ ]:
from sklearn.decomposition import PCA

# First: 2 components, just to visualize
pca_2d = PCA(n_components=2)
X_train_2d = pca_2d.fit_transform(X_train_scaled)

plt.figure(figsize=(7, 5))
sns.scatterplot(x=X_train_2d[:, 0], y=X_train_2d[:, 1],
                hue=y_train.map({0: "malignant", 1: "benign"}), alpha=0.7)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("Breast cancer data in 2D (via PCA)")
plt.show()

print(f"These 2 components already capture "
      f"{pca_2d.explained_variance_ratio_.sum():.1%} of the variation.")

Even in just **2 dimensions** the two classes are largely separated — a good sign that classification
will work well. Now let's choose how many components to keep for the model.

In [ ]:
# How many components do we need for ~95% of the variance?
pca_full = PCA().fit(X_train_scaled)
cumulative = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(7, 4))
plt.plot(range(1, len(cumulative) + 1), cumulative, "o-")
plt.axhline(0.95, color="red", linestyle="--", label="95% variance")
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.title("How many components do we need?")
plt.legend()
plt.show()

n_components = int(np.argmax(cumulative >= 0.95) + 1)
print(f"{n_components} components retain ~95% of the information (down from 30 features).")

In [ ]:
# Apply PCA with the chosen number of components — fit on TRAIN only
pca = PCA(n_components=n_components)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print("Reduced shape:", X_train_pca.shape, "(was 30 features)")

## Milestone 4 — Choose & train a model

This is **binary classification**. We try two simple, beginner-friendly models on the PCA features and
compare them: **Logistic Regression** and **K-Nearest Neighbors**. To check whether PCA actually helped,
we also train Logistic Regression on the *full* 30 scaled features.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Models on PCA-reduced features
logreg_pca = LogisticRegression(max_iter=5000).fit(X_train_pca, y_train)
knn_pca = KNeighborsClassifier(n_neighbors=5).fit(X_train_pca, y_train)

# Baseline: Logistic Regression on all 30 scaled features (no PCA)
logreg_full = LogisticRegression(max_iter=5000).fit(X_train_scaled, y_train)

print(f"Logistic Regression + PCA : {accuracy_score(y_test, logreg_pca.predict(X_test_pca)):.3f}")
print(f"KNN + PCA                 : {accuracy_score(y_test, knn_pca.predict(X_test_pca)):.3f}")
print(f"Logistic Regression (30 features, no PCA): "
      f"{accuracy_score(y_test, logreg_full.predict(X_test_scaled)):.3f}")

Typically all three land around **95–98%**. PCA keeps almost the same accuracy while using **far fewer
features** — that is the win: a simpler, faster model with little to no loss in quality. We pick
**Logistic Regression + PCA** as our final model.

## Milestone 5 — Evaluate the final model

Accuracy alone is not enough for a medical task. We look at the **confusion matrix** and
**precision/recall** to understand *what kind* of mistakes the model makes.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

final_model = logreg_pca
y_pred = final_model.predict(X_test_pca)

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=["malignant", "benign"])
plt.title("Confusion Matrix — Logistic Regression + PCA")
plt.show()

print(classification_report(y_test, y_pred, target_names=["malignant", "benign"]))

**How to read this:** the most important number for this problem is the **recall for the *malignant*
class** — of all the real cancers, how many did the model catch? A false negative (a missed cancer) is the
dangerous error. Encourage students to comment on this rather than just the overall accuracy.

## Milestone 6 — Present the results

A 5-minute presentation should answer: *What was the task, what did you do, and how well did it work?*

**Example summary (what students could say):**

- **Task:** Predict whether a tumor is malignant or benign from 30 cell-measurements (binary classification).
- **Data:** 569 samples, fairly balanced classes, no missing values, but features on very different scales.
- **Preprocessing:** Standardized all features, then used **PCA** to compress 30 correlated features down to
  ~10 components that keep ~95% of the information. PCA in 2D also showed the classes are clearly separable.
- **Model:** Logistic Regression trained on the PCA features.
- **Result:** ~96% accuracy; most importantly, high recall on the malignant class (few missed cancers).
  PCA gave essentially the same accuracy as using all 30 features, with a simpler model.
- **Limitations / next steps:** small dataset; would need more data and clinical validation before real use;
  could try other models (Decision Tree, SVM) or tune the decision threshold to catch even more cancers.

---

### Grading note for instructors

Map this against `demos/PROJECT_RUBRIC.md`: this example covers data exploration, a justified preprocessing
choice (scaling + PCA, explained), an appropriate model for the task type, evaluation **beyond accuracy**
(confusion matrix + recall reasoning), and a clear presentation narrative. A strong student submission does
not need PCA — but it should *justify* its preprocessing and evaluation choices the way this one does.